# 🌙 04. Ground Truth Dataset Preparation & Selenographic Verification

**Mission Context**: Establishing rigorous ground truth tie-point correspondences and known 3x3 projective transformation matrices ($H_{GT}$) for Chandrayaan-2 optical validation.  
**Objectives**:
- Generate synthetic and georeferenced optical image pairs.
- Derive exact ground truth correspondence points $\mathbf{x}' \sim H_{GT} \mathbf{x}$.
- Validate annotation geometric consistency.
- Export `ground_truth.csv` and `ground_truth.json`.


In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import json

sys.path.append(str(Path.cwd().parent))
from lunar_core.config import load_config
from lunar_core.synthetic_data import LunarSyntheticGenerator

config = load_config()
gen = LunarSyntheticGenerator(size=(512, 512), seed=101)
pair = gen.generate_registered_pair(rotation_deg=14.5, scale=1.06, tx=40.0, ty=-25.0)

ref_img = pair["reference_image"]
src_img = pair["source_image"]
H_gt = pair["homography_ground_truth"]
pts_ref = pair["ref_points"]
pts_src = pair["src_points"]

print(f"Generated Ground Truth Pair with {len(pts_ref)} verified tie-points.")
print(f"Ground Truth Homography Matrix:\n{H_gt}")


In [ ]:
# Visualize Correspondence Points and Overlay
h, w = ref_img.shape
vis = np.zeros((h, w * 2, 3), dtype=np.uint8)
vis[:, :w] = cv2.cvtColor(src_img, cv2.COLOR_GRAY2BGR)
vis[:, w:] = cv2.cvtColor(ref_img, cv2.COLOR_GRAY2BGR)

for p_src, p_ref in zip(pts_src[:60], pts_ref[:60]):
    pt1 = (int(p_src[0]), int(p_src[1]))
    pt2 = (int(p_ref[0] + w), int(p_ref[1]))
    cv2.line(vis, pt1, pt2, (0, 255, 255), 1, cv2.LINE_AA)
    cv2.circle(vis, pt1, 3, (0, 0, 255), -1)
    cv2.circle(vis, pt2, 3, (0, 255, 0), -1)

plt.figure(figsize=(16, 8))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"Ground Truth Correspondence Vectors (Sample 60 Points of {len(pts_ref)})", fontsize=14, fontweight='bold')
plt.axis('off')
os.makedirs("outputs/visualizations", exist_ok=True)
plt.savefig("outputs/visualizations/04_ground_truth_correspondences.png", dpi=300)
plt.show()


In [ ]:
# Export ground truth datasets
records = []
for i in range(len(pts_ref)):
    records.append({
        "point_id": i,
        "src_x": round(float(pts_src[i, 0]), 4),
        "src_y": round(float(pts_src[i, 1]), 4),
        "ref_x": round(float(pts_ref[i, 0]), 4),
        "ref_y": round(float(pts_ref[i, 1]), 4),
    })

df_gt = pd.DataFrame(records)
os.makedirs("outputs/reports", exist_ok=True)
df_gt.to_csv("outputs/reports/ground_truth.csv", index=False)

gt_summary = {
    "total_tie_points": len(pts_ref),
    "rotation_deg": pair["rotation_deg"],
    "scale": pair["scale"],
    "translation": pair["translation"],
    "homography_matrix": H_gt.tolist()
}
with open("outputs/reports/ground_truth.json", "w") as f:
    json.dump(gt_summary, f, indent=4)

print("Exported outputs/reports/ground_truth.csv and ground_truth.json")
